# Phase 4: Storage, Drift Detection & Alerts — Theory & Lab
## DynamoDB Design, Configuration Drift, and Real-Time Streaming

**For:** Jon — AWS DVA-C02, DDIA reader, System Design Interview student  
**Estimated Time:** 8–10 hours  
**Learning Outcomes:**
- Understand configuration drift and why it matters for compliance
- Master DynamoDB single-table design patterns
- Design S3 lifecycle policies for evidence retention
- Build a DriftDetector class comparing scan results
- Implement SNS notifications for drift alerts
- Use DynamoDB Streams and EventBridge for real-time detection

**Cross-References:**
- 📘 DDIA Ch. 3: Storage Engines (LSM trees → DynamoDB internals)
- 📘 DDIA Ch. 5: Replication (durability guarantees)
- 📘 DDIA Ch. 11: Stream Processing (EventBridge, real-time drift)
- 📙 System Design Interview Ch. 5: Consistent Hashing (DynamoDB partitioning)
- 📙 System Design Interview Ch. 6: Design a Key-Value Store (DynamoDB patterns)
- 🎓 AWS DVA-C02: Domain 2 (DynamoDB, Streams), Domain 4 (Troubleshooting)

---

---
## How This Phase Fits Into Everything You've Built

By now you have a complete data pipeline. Phase 4 adds two new concerns: **persistence** (saving results so you can compare them later) and **change detection** (finding what changed between two snapshots).

```
Phase 1                  Phase 2                  Phase 3              Phase 4 ← HERE
──────────────           ──────────────────        ──────────────────   ─────────────────────────
boto3 API calls          ControlMappingEngine      PDFReportGenerator   DriftDetector
  ↓                        ↓                         ↓                    ↓
EvidenceItem[]           ControlAssessment[]       PDF on S3            DriftEvent[]
  ↓                        ↓                         ↓                    ↓
ScanResult               CompliancePosture         Presigned URL        SNS Alert
                                                                           ↓
                                                                        DynamoDB
                                                                        (history)
```

**The new concept this phase introduces: state over time.**

Phases 1–3 answered: *"What is the compliance posture right now?"*  
Phase 4 answers: *"Did anything change since last time?"*

To answer that question you need:
1. **Storage** — persist each scan's results (DynamoDB)
2. **Comparison** — diff previous vs current (DriftDetector)
3. **Alerting** — notify when something regressed (SNS)
4. **Real-time triggers** — detect changes as they happen (DynamoDB Streams / EventBridge)

**The data types you already know (`ControlAssessment`, `CompliancePosture`) are unchanged.** Phase 4 adds exactly two new types: `DriftEvent` (what changed) and the DynamoDB storage layer (where it's saved). Everything else is composition of patterns you've already seen.

---

## Part 1: Theory — Configuration Drift

### What is Configuration Drift?

**Configuration drift** = When actual deployed infrastructure diverges from the desired state.

**Example scenarios:**

| Event | Desired State | Actual State | Drift? |
|-------|---------------|--------------|--------|
| Deploy S3 bucket with versioning | Versioning ON | Versioning ON | ✓ NO |
| Engineer manually disables versioning | Versioning ON | Versioning OFF | ✗ YES |
| Patch updates an IAM policy | 5 permissions | 6 permissions | ✗ YES (new permission) |
| Auto-scaling adds servers | 3-5 servers | 5 servers | ✓ NO |
| Someone adds exception to Security Group | Port 22 blocked | Port 22 open | ✗ YES |

### Why Does Drift Matter for Compliance?

Auditors (SOC 2, FedRAMP, HIPAA) want to know: **Did you implement the controls you said you implemented?**

**Problem:** If drift happens and you don't detect it:
- You claim "All S3 buckets have versioning" → But one doesn't → Control fails
- You claim "MFA required for root" → But root MFA was disabled → Control fails
- You claim "All logs shipped to immutable archive" → But one region broke → Control fails

**Solution:** **Continuous compliance monitoring** = Detect drift as soon as it happens.

### Detection Strategies

**1. Periodic Scan (Batch)**
```
Every 1 hour:
  scan entire AWS account → DynamoDB
  compare to previous scan
  find differences
  alert if drift detected
```
**Pros:** Simple, catch all changes  
**Cons:** Delayed detection (up to 1 hour), high API costs

**2. Event-Driven (Streaming)**
```
AWS Config Change → EventBridge → Lambda → Compare → SNS Alert
```
**Pros:** Real-time, low cost (pay per event)  
**Cons:** Must configure for every resource type

**3. Hybrid (Recommended)**
```
Event-driven alerts (real-time) + Periodic scan (catch gaps)
```

---

---
### Plain English: What Drift Detection Is Actually Doing

Strip away all the cloud terminology. At its core, drift detection is a **diff between two snapshots**.

```python
# This is conceptually all DriftDetector.detect() does:
def detect(previous: list, current: list) -> list:
    events = []
    for control_id in all_control_ids:
        prev_status = find_status(previous, control_id)
        curr_status = find_status(current, control_id)
        if prev_status != curr_status:
            events.append(DriftEvent(control_id, prev_status, curr_status))
    return events
```

This same pattern appears everywhere in engineering:

| Tool / System | "Previous" | "Current" | "Diff" |
|---|---|---|---|
| `git diff` | last commit | working tree | changed lines |
| Database CDC | old row image | new row image | changed columns |
| Terraform plan | current infra state | desired state | resources to change |
| DriftDetector | last scan assessments | new scan assessments | changed control statuses |
| React Virtual DOM | previous render | next render | DOM patches to apply |

**DDIA Connection (Ch. 11 — Stream Processing, p. 455):** Kleppmann calls this **Change Data Capture (CDC)** — "the process of observing all data changes written to a database and extracting them in a form in which they can be replicated to other systems." Our drift detector is a CDC implementation: it reads two snapshots of the assessment table and emits change events.

**Why "drift" instead of just "change"?**  
"Drift" implies the change was unintended — infrastructure moved away from a desired state without anyone explicitly requesting it. A developer accidentally leaves a security group rule open. An auto-remediation job fails silently. A third-party service modifies an IAM policy. These are all drift. A deliberate deployment that changes a control status is not drift — but our detector can't distinguish intent from accident, so it reports all status changes and lets humans decide.

---

## Part 2: DynamoDB Single-Table Design Deep Dive

### Traditional Multi-Table Design (Anti-Pattern)

```
Table: Controls
┌─────────────────┐
│ control_id (PK) │
│ name            │
│ status          │
└─────────────────┘

Table: Findings
┌──────────────────┐
│ finding_id (PK)  │
│ control_id (FK)  │ ← Requires JOIN
│ severity         │
└──────────────────┘

Table: ScanResults
┌──────────────────────┐
│ scan_id (PK)         │
│ control_id (FK)      │ ← Requires JOIN
│ timestamp            │
└──────────────────────┘
```

**Problems:**
- Multiple round-trips to DynamoDB
- No native JOINs (need application logic)
- Transaction complexity
- Harder to scale

### Single-Table Design (Pattern)

```
Table: ComplianceData

PK: entity_type#entity_id
SK: sort_key (context-dependent)
GSI1_PK: timestamp#control_id
GSI1_SK: scan_status
```

**Example data:**

```
PK                      SK                  Type     Status    Data
─────────────────────────────────────────────────────────────────
CONTROL#AC-2            METADATA            Control  COMPLIANT {...}
CONTROL#AC-2            FINDING#F-001       Control  NON_COMP  {...}
CONTROL#AC-2            SCAN#2026-05-03T15  Control  LATEST    {...}

SCAN#2026-05-03T15      CONTROL#AC-2        Scan     OK        {...}
SCAN#2026-05-03T15      CONTROL#AC-3        Scan     OK        {...}

DRIFT#AC-2              2026-05-03T15:30    Drift    CRITICAL  {...}
```

### Key Design Decisions

**1. Primary Key (PK & SK)**

| Entity | PK | SK | Use Case |
|--------|-----|-----|----------|
| Control | `CONTROL#AC-2` | `METADATA` | Get control definition |
| Control | `CONTROL#AC-2` | `FINDING#F-001` | Query findings for control |
| Control | `CONTROL#AC-2` | `SCAN#2026-05-03` | Historical scan results |
| Scan | `SCAN#2026-05-03T15` | `CONTROL#AC-2#DIFF` | What changed in this scan? |
| Drift Event | `DRIFT#AC-2` | `2026-05-03T15:30` | Drift timeline for control |

**2. Global Secondary Index (GSI) for Time-Series Queries**

```
GSI1:
  PK: timestamp (e.g., "2026-05")
  SK: control_id#severity (e.g., "AC-2#HIGH")
  
Queries:
  "Show me all findings from May 2026" → Query GSI1 PK = "2026-05"
  "Show me HIGH severity findings from May" → Query GSI1 with SK filter
```

**3. Attributes for Drift Detection**

```json
{
  "PK": "CONTROL#AC-2",
  "SK": "SCAN#2026-05-03T15:30",
  "current_status": "COMPLIANT",
  "previous_status": "COMPLIANT",
  "status_changed": false,
  "findings_count": 0,
  "findings_count_previous": 0,
  "drift_detected": false,
  "drift_type": null,
  "timestamp": 1714849800,
  "ttl": 1746384000  # Data expires after 7 years
}
```

---

---
### DynamoDB Single-Table Design: Why It Feels Backwards (And Why It's Right)

If you've used SQL, single-table design will feel wrong at first. Here's how to reframe it.

**In SQL, you organize by *data type*:**
```sql
CREATE TABLE controls (control_id, title, family)
CREATE TABLE scans     (scan_id, timestamp, account_id)
CREATE TABLE findings  (finding_id, control_id, severity)
-- Then JOIN them at query time
```

**In DynamoDB, you organize by *access pattern*:**
```
Single table: compliance-assessments
  PK                    SK                       → answers the question:
  CTRL#AC-2             SCAN#2024-01-15          "What was AC-2's status on Jan 15?"
  CTRL#AC-2             SCAN#2024-01-08          "What was AC-2's status on Jan 8?"
  SCAN#2024-01-15       CTRL#AC-2                "What controls were assessed on Jan 15?"
  DRIFT#SCAN#2024-01-15 CTRL#AC-2#drift-abc123   "What drifted in the Jan 15 scan?"
  POSTURE               SCAN#2024-01-15          "What was the overall posture on Jan 15?"
```

The same data appears under *different keys* depending on how you need to access it. This feels like duplication but is actually the DynamoDB equivalent of a database index.

**The mental model shift:**
```
SQL thought: "What table does this data belong in?"
DynamoDB thought: "What question will I ask to retrieve this data?"
```

**DDIA Connection (Ch. 2 — Data Models, p. 38):** Kleppmann notes that document databases trade normalization for locality — "all the relevant data is in one place." Single-table DynamoDB takes this further: all access patterns are served from one table, with keys designed per query rather than per entity type.

**When does this NOT work?**  
Ad-hoc queries. If a new stakeholder asks "show me all CRITICAL findings across all scans in Q1," and you didn't design a key for that, you need a `Scan` (full table read, expensive). The trade-off: DynamoDB is fast and cheap for *designed* access patterns, expensive for *unexpected* ones. SQL is flexible but slower at scale. For a known, stable set of queries (which compliance reports are), DynamoDB wins.

**The three access patterns we design for:**
```
1. "Get all controls in scan X"          → Query GSI1 PK=SCAN#{scan_id}
2. "Get history of control AC-2"         → Query PK=CTRL#AC-2, SK begins_with SCAN#
3. "Get all drift in scan X"             → Query PK=DRIFT#SCAN#{scan_id}
```
Every PK/SK decision in the lab below serves one of these three queries.

---

## Part 3: S3 Lifecycle Policies for Evidence Retention

### Compliance Retention Requirements

| Framework | Retention Period | Rationale |
|-----------|------------------|----------|
| **SOC 2** | 1 year minimum | Annual audit cycles |
| **FedRAMP** | 3 years active + 1 year archive | Federal records management |
| **HIPAA** | 6 years minimum | HHS audit requirements |
| **PCI DSS** | 1 year min + 3 months online | Card brand requirements |
| **Internal Best Practice** | 7 years | Statute of limitations |

### S3 Lifecycle Strategy

```
Day 0: Create Evidence PDF
  └─ Storage Class: STANDARD (fast access)
     Cost: $0.023 per GB/month

Day 90: Archive to Glacier
  └─ Storage Class: GLACIER (cold)
     Cost: $0.004 per GB/month (80% savings)
     Retrieval: 1-5 minutes

Day 180: Archive to Deep Archive
  └─ Storage Class: DEEP_ARCHIVE (frozen)
     Cost: $0.00099 per GB/month (95% savings)
     Retrieval: 12 hours

Day 2555 (7 years): Delete
  └─ Automatically removed
```

### Terraform Configuration

```hcl
resource "aws_s3_bucket_lifecycle_configuration" "evidence" {
  bucket = aws_s3_bucket.evidence.id

  rule {
    id     = "ArchiveOldEvidence"
    status = "Enabled"
    filter {
      prefix = "reports/"
    }

    # Transition to Glacier after 90 days
    transition {
      days          = 90
      storage_class = "GLACIER"
    }

    # Transition to Deep Archive after 180 days
    transition {
      days          = 180
      storage_class = "DEEP_ARCHIVE"
    }

    # Delete after 7 years
    expiration {
      days = 2555
    }
  }
}
```

---

## Part 4: DDIA & System Design Interview Tie-Ins

### DDIA Ch. 3: Storage Engines — LSM Trees

**How does DynamoDB work under the hood?**

DynamoDB uses **LSM (Log-Structured Merge) trees** for writes:

```
Write Request
  ↓
In-Memory Buffer (Memtable)
  ↓ (when full)
Write to Disk (SSTable - Sorted String Table)
  ↓
Compact Multiple SSTables
  ↓
Read merges multiple levels
```

**Why?**
- Sequential writes (fast on disk)
- Compaction (cleanup old versions)
- Efficient range queries (sorted)
- Built-in TTL support (DynamoDB deletes expired items during compaction)

### DDIA Ch. 5: Replication — Durability & Consistency

DynamoDB guarantees:
- **Multi-AZ replication** (3 replicas automatically)
- **Durable writes** (written to disk before acknowledgement)
- **Eventual consistency** by default (fast but slightly stale reads)
- **Strong consistency** available (slower but current)

**For drift detection:**
```python
# Get consistent current state (not stale)
response = dynamodb.get_item(
    Key={'PK': 'CONTROL#AC-2', 'SK': 'SCAN#2026-05-03'},
    ConsistentRead=True  # ← Important for comparing
)
```

### DDIA Ch. 11: Stream Processing — Real-Time Drift

**DynamoDB Streams** capture every write in order:

```
DynamoDB Write
  ↓
Trigger DynamoDB Stream
  ↓
Stream Consumer (Lambda)
  ↓
Analyze old → new values
  ↓
If drift: Publish to SNS
```

### System Design Interview Ch. 5: Consistent Hashing

**How does DynamoDB partition data?**

DynamoDB uses consistent hashing on the partition key:

```
Partition Ring:
  0 ━━━━━━━━━━━━━━━━━ 4294967296
  │                      │
  ├─ Partition 1 ────────┤
  ├─ Partition 2 ────────┤
  └─ Partition 3 ────────┘

Hash("CONTROL#AC-2") = 1234567890 → Falls in Partition 2
```

**For drift detection:**
```python
# Bad partition key = hot partition
PK = "CONTROL#AC-2"  # All controls hash to same partition!

# Good partition key = distributed
PK = "timestamp#CONTROL#AC-2"  # Spread across time
```

### System Design Interview Ch. 6: Design a Key-Value Store

**DynamoDB IS a key-value store:**

```
Key:   (PK, SK)
Value: {status, findings, timestamp, ...}

Operations:
  PUT    - Write a control assessment
  GET    - Read latest status
  QUERY  - Get all assessments for a control
  SCAN   - Read everything (expensive)
```

---

## Part 5: DVA-C02 Tie-Ins

### Domain 2: DynamoDB Operations

**Core operations tested:**

| Operation | Syntax | Use Case |
|-----------|--------|----------|
| `put_item()` | Write one item | Store scan result |
| `get_item()` | Read by PK+SK | Get latest control status |
| `query()` | All items with PK | Get all findings for control |
| `scan()` | All items (expensive) | Full table export |
| `batch_write_item()` | Write 25 items at once | Bulk import scan results |
| `batch_get_item()` | Read 100 items at once | Fetch multiple controls |
| `update_item()` | Modify existing item | Increment finding count |

### Domain 4: DynamoDB TTL & Monitoring

**TTL (Time To Live):**
```python
# Automatically delete scan results after 7 years
item = {
    'PK': 'SCAN#2026-05-03',
    'SK': 'CONTROL#AC-2',
    'ttl': int((datetime.now() + timedelta(days=2555)).timestamp())
}
```

**Monitoring:**
- CloudWatch metrics: Read/Write capacity, throttling
- DynamoDB Streams: Monitor changes
- DAX: Caching layer for hot data

---

## 📚 Documentation Links

### DynamoDB
- [DynamoDB Developer Guide](https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/)
- [DynamoDB Streams](https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/Streams.html)
- [boto3 DynamoDB Resource](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/dynamodb.html)
- [Single-Table Design Pattern](https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/workbench.html) (DynamoDB Workbench guide)

### AWS Config & EventBridge
- [AWS Config Recorder](https://docs.aws.amazon.com/config/latest/developerguide/manage-config.html)
- [EventBridge Rules](https://docs.aws.amazon.com/eventbridge/latest/userguide/eb-rules.html)
- [EventBridge Patterns](https://docs.aws.amazon.com/eventbridge/latest/userguide/eb-event-patterns.html)

### S3 Lifecycle
- [S3 Lifecycle Configuration](https://docs.aws.amazon.com/AmazonS3/latest/userguide/object-lifecycle-mgmt.html)
- [S3 Storage Classes](https://docs.aws.amazon.com/AmazonS3/latest/userguide/storage-class-intro.html)

### Terraform
- [aws_dynamodb_table](https://registry.terraform.io/providers/hashicorp/aws/latest/docs/resources/dynamodb_table)
- [aws_s3_bucket_lifecycle_configuration](https://registry.terraform.io/providers/hashicorp/aws/latest/docs/resources/s3_bucket_lifecycle_configuration)

---

# Lab: Building Real-Time Drift Detection

## Objective

Build a **DriftDetector** class that:
1. Compares current vs previous scan results in DynamoDB
2. Identifies configuration changes (drift)
3. Publishes alerts to SNS
4. Tracks drift history for compliance reporting

## Lab Structure

- **Lab 4.1:** DynamoDB single-table design and boto3 operations
- **Lab 4.2:** Drift comparison logic
- **Lab 4.3:** SNS alerting
- **Lab 4.4:** DynamoDB Streams integration
- **Lab 4.5:** EventBridge for real-time detection

---

## Lab 4.1: DynamoDB Single-Table Design

**Goal:** Design and create a DynamoDB table that stores control assessments, scans, and drift events.


In [2]:
# Import everything from src/ — no inline redefinitions
import sys, os, json
sys.path.insert(0, os.path.abspath('../..'))

from src.models import (
    EvidenceItem, ScanResult, CollectorResult, ControlAssessment,
    ControlStatus, CompliancePosture, DriftEvent,
    generate_scan_id, generate_drift_id,
    SEVERITY_WEIGHTS, CONTROL_FAMILIES
)
from src.mapper.control_catalog import NIST_CONTROL_CATALOG
from src.mapper.engine import ControlMappingEngine
from src.drift.detector import DriftDetector

# Show canonical types from src/models.py
print('ControlStatus values (from src/models.py):')
for status in ControlStatus:
    print(f'  {status.name} = {status.value}')

print(f'\nDriftEvent fields: {list(DriftEvent.__dataclass_fields__.keys())}')
print(f'Loaded {len(NIST_CONTROL_CATALOG)} NIST controls')
print(f'DriftDetector methods: detect(previous, current) -> List[DriftEvent]')

ControlStatus values (from src/models.py):
  PASS = PASS
  FAIL = FAIL
  PARTIAL = PARTIAL
  NOT_ASSESSED = NOT_ASSESSED
  NOT_APPLICABLE = N/A

DriftEvent fields: ['drift_id', 'control_id', 'control_title', 'previous_status', 'current_status', 'drift_type', 'severity', 'previous_scan_id', 'current_scan_id', 'timestamp', 'affected_resources', 'details']
Loaded 25 NIST controls
DriftDetector methods: detect(previous, current) -> List[DriftEvent]


---
### New Imports in Phase 4 — What Each One Does

```python
from src.models import DriftEvent, generate_drift_id
```

**`DriftEvent`** — the new type this phase introduces. Fields:
```
drift_id          unique identifier (generated by generate_drift_id())
control_id        which NIST control changed (e.g., "AC-2")
control_title     human name (e.g., "Account Management")
previous_status   ControlStatus before the change (e.g., "FAIL")
current_status    ControlStatus after the change (e.g., "PASS")
drift_type        "REGRESSION" | "IMPROVEMENT" | "STATUS_CHANGE" | "NEW_FINDING" | "RESOLVED"
severity          severity of the event (CRITICAL for REGRESSION, LOW for IMPROVEMENT)
is_regression     bool — True if the control got worse
affected_resources list of resource ARNs involved in the change
details           human-readable explanation
previous_scan_id  scan ID of the "before" snapshot
current_scan_id   scan ID of the "after" snapshot
timestamp         when drift was detected (ISO 8601 UTC)
```

```python
from src.drift.detector import DriftDetector
```

**`DriftDetector`** — wraps the diff logic. `detector.detect(prev_assessments, curr_assessments) -> List[DriftEvent]`. Internally it builds lookup dicts by `control_id` and compares statuses — exactly the pattern shown in the theory section above.

**Why is drift detection in `src/drift/` and not `src/mapper/`?**  
Separation of concerns. The mapper turns evidence into assessments (Phase 2 concern). The drift detector compares assessments across time (Phase 4 concern). Keeping them separate means you can test each independently and swap out either implementation without breaking the other.

---

In [3]:
# Show the Terraform configuration that would create this table
terraform_config = '''
resource "aws_dynamodb_table" "compliance_data" {
  name           = "compliance-assessments"
  billing_mode   = "PAY_PER_REQUEST"  # or "PROVISIONED"
  hash_key       = "PK"
  range_key      = "SK"

  attribute {
    name = "PK"
    type = "S"
  }

  attribute {
    name = "SK"
    type = "S"
  }

  attribute {
    name = "timestamp"
    type = "S"
  }

  # Global Secondary Index for time-series queries
  global_secondary_index {
    name            = "timestamp-index"
    hash_key        = "timestamp"
    projection_type = "ALL"
  }

  # TTL for automatic cleanup
  ttl {
    attribute_name = "ttl"
    enabled        = true
  }

  # Enable DynamoDB Streams for real-time processing
  stream_specification {
    stream_view_type = "NEW_AND_OLD_IMAGES"
  }
}
'''

print("Terraform configuration for DynamoDB table:")
print(terraform_config)

Terraform configuration for DynamoDB table:

resource "aws_dynamodb_table" "compliance_data" {
  name           = "compliance-assessments"
  billing_mode   = "PAY_PER_REQUEST"  # or "PROVISIONED"
  hash_key       = "PK"
  range_key      = "SK"

  attribute {
    name = "PK"
    type = "S"
  }

  attribute {
    name = "SK"
    type = "S"
  }

  attribute {
    name = "timestamp"
    type = "S"
  }

  # Global Secondary Index for time-series queries
  global_secondary_index {
    name            = "timestamp-index"
    hash_key        = "timestamp"
    projection_type = "ALL"
  }

  # TTL for automatic cleanup
  ttl {
    attribute_name = "ttl"
    enabled        = true
  }

  # Enable DynamoDB Streams for real-time processing
  stream_specification {
    stream_view_type = "NEW_AND_OLD_IMAGES"
  }
}



### Building Two Scans for Drift Comparison

To demonstrate DynamoDB storage and drift detection, we need two scans at different points in time.
We build them using the same pipeline as Phases 2–3: evidence → ScanResult → ControlMappingEngine → ControlAssessment[].

In [4]:
# ---- Scan 1: Baseline (some failures) ----
baseline_evidence = [
    EvidenceItem(source='security_hub', finding_id='sh-iam4-001',
        title='IAM.4 Root MFA not enabled', status='FAILED', severity='CRITICAL',
        resource_type='AWS::IAM::User', resource_id='arn:aws:iam::123456789012:root',
        timestamp='2024-01-08T10:00:00Z', remediation='Enable hardware MFA on root',
        control_ids=['AC-2', 'IA-2', 'IA-2(1)']),
    EvidenceItem(source='security_hub', finding_id='sh-s3-005',
        title='S3.5 Buckets require SSL', status='FAILED', severity='HIGH',
        resource_type='AWS::S3::Bucket', resource_id='arn:aws:s3:::prod-data',
        timestamp='2024-01-08T10:01:00Z', remediation='Add SecureTransport policy',
        control_ids=['SC-8', 'SC-13']),
    EvidenceItem(source='config', finding_id='cfg-ct-001',
        title='multi-region-cloudtrail-enabled: COMPLIANT', status='PASSED',
        severity='INFORMATIONAL', resource_type='AWS::CloudTrail::Trail',
        resource_id='arn:aws:cloudtrail:us-east-1:trail/org-trail',
        timestamp='2024-01-08T10:02:00Z', control_ids=['AU-2', 'AU-3', 'AU-12']),
    EvidenceItem(source='config', finding_id='cfg-gd-001',
        title='guardduty-enabled: COMPLIANT', status='PASSED', severity='INFORMATIONAL',
        resource_type='AWS::GuardDuty::Detector', resource_id='detector-us-east-1',
        timestamp='2024-01-08T10:03:00Z', control_ids=['SI-4']),
    EvidenceItem(source='config', finding_id='cfg-ssh-001',
        title='restricted-ssh: COMPLIANT', status='PASSED', severity='INFORMATIONAL',
        resource_type='AWS::EC2::SecurityGroup', resource_id='sg-0abc123',
        timestamp='2024-01-08T10:04:00Z', control_ids=['SC-7', 'CM-6']),
    EvidenceItem(source='security_hub', finding_id='sh-iam1-001',
        title='IAM.1 No wildcard admin policies', status='PASSED', severity='INFORMATIONAL',
        resource_type='AWS::IAM::Policy', resource_id='arn:aws:iam::123456789012:policy/Dev',
        timestamp='2024-01-08T10:08:00Z', control_ids=['AC-6', 'AC-3']),
]

# ---- Scan 2: Post-remediation (root MFA fixed, new EBS issue) ----
remediated_evidence = [
    EvidenceItem(source='security_hub', finding_id='sh-iam4-001',
        title='IAM.4 Root MFA enabled', status='PASSED', severity='INFORMATIONAL',
        resource_type='AWS::IAM::User', resource_id='arn:aws:iam::123456789012:root',
        timestamp='2024-01-15T10:00:00Z', control_ids=['AC-2', 'IA-2', 'IA-2(1)']),
    EvidenceItem(source='security_hub', finding_id='sh-s3-005',
        title='S3.5 Buckets require SSL', status='PASSED', severity='INFORMATIONAL',
        resource_type='AWS::S3::Bucket', resource_id='arn:aws:s3:::prod-data',
        timestamp='2024-01-15T10:01:00Z', control_ids=['SC-8', 'SC-13']),
    EvidenceItem(source='config', finding_id='cfg-ct-001',
        title='multi-region-cloudtrail-enabled: COMPLIANT', status='PASSED',
        severity='INFORMATIONAL', resource_type='AWS::CloudTrail::Trail',
        resource_id='arn:aws:cloudtrail:us-east-1:trail/org-trail',
        timestamp='2024-01-15T10:02:00Z', control_ids=['AU-2', 'AU-3', 'AU-12']),
    EvidenceItem(source='config', finding_id='cfg-gd-001',
        title='guardduty-enabled: COMPLIANT', status='PASSED', severity='INFORMATIONAL',
        resource_type='AWS::GuardDuty::Detector', resource_id='detector-us-east-1',
        timestamp='2024-01-15T10:03:00Z', control_ids=['SI-4']),
    EvidenceItem(source='config', finding_id='cfg-ssh-001',
        title='restricted-ssh: COMPLIANT', status='PASSED', severity='INFORMATIONAL',
        resource_type='AWS::EC2::SecurityGroup', resource_id='sg-0abc123',
        timestamp='2024-01-15T10:04:00Z', control_ids=['SC-7', 'CM-6']),
    EvidenceItem(source='security_hub', finding_id='sh-iam1-001',
        title='IAM.1 No wildcard admin policies', status='PASSED', severity='INFORMATIONAL',
        resource_type='AWS::IAM::Policy', resource_id='arn:aws:iam::123456789012:policy/Dev',
        timestamp='2024-01-15T10:08:00Z', control_ids=['AC-6', 'AC-3']),
    EvidenceItem(source='config', finding_id='cfg-ebs-001',
        title='encrypted-volumes: NON_COMPLIANT', status='FAILED', severity='HIGH',
        resource_type='AWS::EC2::Volume', resource_id='vol-0abc123',
        timestamp='2024-01-15T10:06:00Z', remediation='Enable EBS default encryption',
        control_ids=['SC-13', 'SC-28']),
]

# Run both through the pipeline (same as Phases 2-3)
engine = ControlMappingEngine()

scan1 = ScanResult(scan_id='2024-01-08T10-00-00Z_baseline',
                   scan_start='2024-01-08T10:00:00Z',
                   account_id='123456789012', region='us-east-1')
cr1 = CollectorResult(source='all_collectors', status='SUCCESS',
                      evidence_items=baseline_evidence,
                      raw_findings_count=len(baseline_evidence))
scan1.collector_results['all'] = cr1
scan1.finalize()
assessments1 = engine.assess_all_controls(scan1)
posture1 = engine.generate_posture(assessments1)

scan2 = ScanResult(scan_id='2024-01-15T10-00-00Z_remediated',
                   scan_start='2024-01-15T10:00:00Z',
                   account_id='123456789012', region='us-east-1')
cr2 = CollectorResult(source='all_collectors', status='SUCCESS',
                      evidence_items=remediated_evidence,
                      raw_findings_count=len(remediated_evidence))
scan2.collector_results['all'] = cr2
scan2.finalize()
assessments2 = engine.assess_all_controls(scan2)
posture2 = engine.generate_posture(assessments2)

print(f'Scan 1 (baseline):    {posture1.compliance_percentage:.1f}% compliance')
print(f'  PASS: {posture1.passed}  FAIL: {posture1.failed}  PARTIAL: {posture1.partial}')
print(f'Scan 2 (remediated):  {posture2.compliance_percentage:.1f}% compliance')
print(f'  PASS: {posture2.passed}  FAIL: {posture2.failed}  PARTIAL: {posture2.partial}')

Scan 1 (baseline):    32.0% compliance
  PASS: 8  FAIL: 5  PARTIAL: 0
Scan 2 (remediated):  48.0% compliance
  PASS: 12  FAIL: 1  PARTIAL: 1


---
### Pattern Recognition: Two Scans = Before / After = Diff

Study the structure of the code you just ran. It follows a pattern you'll use any time you need to detect change between two states:

```
State A (baseline)           Transformation            State B (current)
──────────────               ────────────────          ──────────────────
baseline_evidence[]    →→→   Phase 1+2 pipeline  →→→  remediated_evidence[]
scan1 + assessments1                                   scan2 + assessments2
        │                                                      │
        └──────────────── DriftDetector.detect() ─────────────┘
                                    │
                              DriftEvent[]
```

**What changed between scan1 and scan2 in the mock data:**
```
AC-2:   FAIL → PASS    (root MFA was fixed)
IA-2:   FAIL → PASS    (same root finding cleared)
IA-2(1): FAIL → PASS   (same root finding cleared)
SC-8:   FAIL → PASS    (S3 SSL was fixed)
SC-13:  FAIL → FAIL    (EBS still unencrypted + new EBS finding)
SC-28:  N/A  → FAIL    (new EBS finding added)
```

These are the exact `DriftEvent` objects the detector will produce in Lab 4.2.

**Important: each scan is a complete snapshot, not a delta.** We don't store "what changed." We store "what the full state was." The detector computes the diff at comparison time. This is the same pattern as:
- **Git:** stores full snapshots of every file, computes diffs on demand
- **Terraform state:** stores complete infrastructure state, plans show diffs
- **S3 versioning:** stores full object at each version, not just the changes

DDIA calls this "immutable, append-only logs" (Ch. 10, p. 459). Each scan result is an immutable record of the world at a point in time. You can always recompute drift by re-diffing any two snapshots.

---

In [6]:
# Show how ControlAssessment maps to DynamoDB single-table items
# This is the bridge between our Python models and DynamoDB storage

ac2 = next(a for a in assessments1 if a.control_id == 'AC-2')
item_dict = ac2.to_dict()

# In DynamoDB single-table design, we add PK/SK keys
dynamodb_item = {
    'PK': f'CTRL#{ac2.control_id}',
    'SK': f'SCAN#{ac2.scan_id}',
    'GSI1PK': f'SCAN#{ac2.scan_id}',
    'GSI1SK': f'CTRL#{ac2.control_id}#{ac2.status.value}',
    **{k: v for k, v in item_dict.items() if k != 'evidence'},  # evidence stored separately
    'evidence_count': len(ac2.evidence),
    'ttl': 1746384000,  # 7-year retention
}

print('=== ControlAssessment -> DynamoDB Item ===')
print(f'PK:     {dynamodb_item["PK"]}')
print(f'SK:     {dynamodb_item["SK"]}')
print(f'GSI1PK: {dynamodb_item["GSI1PK"]}')
print(f'GSI1SK: {dynamodb_item["GSI1SK"]}')
print(f'status: {dynamodb_item["status"]}')
print(f'control_family: {dynamodb_item["control_family"]}')
print(f'highest_severity: {dynamodb_item["highest_severity"]}')
print(f'remediation_priority: {dynamodb_item["remediation_priority"]}')

# Show CompliancePosture as a DynamoDB item too
posture_item = {
    'PK': 'POSTURE',
    'SK': f'SCAN#{posture1.scan_id}',
    **posture1.to_dict(),
}
print(f'\n=== CompliancePosture -> DynamoDB Item ===')
print(f'PK: {posture_item["PK"]}')
print(f'SK: {posture_item["SK"]}')
print(f'compliance_percentage: {posture_item["compliance_percentage"]}')
print(f'passed: {posture_item["passed"]}, failed: {posture_item["failed"]}')

# Show all access patterns
print(f'\n=== Access Patterns ===')
print(f'Get posture:         GetItem(PK=POSTURE, SK=SCAN#{{scan_id}})')
print(f'Get one control:     GetItem(PK=CTRL#AC-2, SK=SCAN#{{scan_id}})')
print(f'All controls in scan: Query(GSI1PK=SCAN#{{scan_id}})')
print(f'Control history:     Query(PK=CTRL#AC-2, SK begins_with SCAN#)')
print(f'Failed controls:     Query(GSI1PK=SCAN#{{scan_id}}) + Filter status=FAIL')

=== ControlAssessment -> DynamoDB Item ===
PK:     CTRL#AC-2
SK:     SCAN#2024-01-08T10-00-00Z_baseline
GSI1PK: SCAN#2024-01-08T10-00-00Z_baseline
GSI1SK: CTRL#AC-2#FAIL
status: FAIL
control_family: AC
highest_severity: CRITICAL
remediation_priority: 10

=== CompliancePosture -> DynamoDB Item ===
PK: POSTURE
SK: SCAN#2024-01-08T10-00-00Z_baseline
compliance_percentage: 32.0
passed: 8, failed: 5

=== Access Patterns ===
Get posture:         GetItem(PK=POSTURE, SK=SCAN#{scan_id})
Get one control:     GetItem(PK=CTRL#AC-2, SK=SCAN#{scan_id})
All controls in scan: Query(GSI1PK=SCAN#{scan_id})
Control history:     Query(PK=CTRL#AC-2, SK begins_with SCAN#)
Failed controls:     Query(GSI1PK=SCAN#{scan_id}) + Filter status=FAIL


---
### Line-by-Line: The DynamoDB Item Structure

The code above did something critical — it translated a Python `ControlAssessment` into a DynamoDB item. Every design decision in that translation has a reason.

**`PK = "CTRL#AC-2"`** — Why the `CTRL#` prefix?  
DynamoDB is a single table. Without prefixes, `PK="AC-2"` could collide with `PK="AC-2"` from a completely different entity type (hypothetically a scan with ID "AC-2"). The `CTRL#` prefix namespaces the key. This is a **universal convention** in single-table design.

**`SK = "SCAN#2024-01-08T10-00-00Z_baseline"`** — Why the scan ID in the sort key?  
Because sort keys are sortable. `Query(PK="CTRL#AC-2", SK between "SCAN#2024-01-01" and "SCAN#2024-02-01")` gives you all assessments of AC-2 in January 2024. ISO 8601 timestamps sort lexicographically, which means alphabetical order = chronological order. This is why the scan ID is formatted `YYYY-MM-DDTHH-MM-SSZ` (dashes instead of colons — colons are reserved in some systems).

**`GSI1PK = "SCAN#2024-01-08T10-00-00Z_baseline"`** — Why duplicate the scan ID as a GSI key?  
The main table answers: "get all assessments FOR A CONTROL across time" (query by PK=CTRL#AC-2).  
The GSI answers: "get all assessments IN A SCAN" (query by GSI1PK=SCAN#{scan_id}).  
Without the GSI, answering "show me all 25 controls for this scan" would require 25 separate `GetItem` calls. With the GSI, it's one `Query` call.

**`**{k: v for k, v in item_dict.items() if k != 'evidence'}`** — What is `**` doing here?

```python
# ** (double-star) unpacks a dict into keyword arguments or merges dicts
base = {"PK": "CTRL#AC-2", "SK": "SCAN#abc"}
extra = {"status": "FAIL", "priority": 10}
merged = {**base, **extra}
# Result: {"PK": "CTRL#AC-2", "SK": "SCAN#abc", "status": "FAIL", "priority": 10}
```
It's dict spread (same as JavaScript's `...spread`). Used here to merge PK/SK keys with all the assessment fields in one expression. The `if k != 'evidence'` filter drops the evidence list — evidence items are stored separately (they could be large, and DynamoDB has a 400KB item size limit).

**`ttl = 1746384000`** — this is a Unix timestamp (seconds since Jan 1, 1970). DynamoDB reads the `ttl` attribute and automatically deletes the item when `now() > ttl`. The item doesn't disappear instantly — DynamoDB's compaction process (the LSM tree compaction from DDIA Ch. 3) cleans it up within 48 hours of expiration. For compliance purposes, treat TTL as "will be gone within 2 days of the date."

---

### Exercise 4.1: Design DynamoDB Items for DriftEvents

Using the single-table pattern above, design the PK/SK structure for storing `DriftEvent` objects.

Consider:
1. What queries does the dashboard need? (e.g., "all drift events for a scan", "drift history for a control")
2. What should the PK be? The SK?
3. Do you need a GSI? What would the GSI key be?

**Hint:** Look at `DriftEvent.__dataclass_fields__` — the key fields are `control_id`, `current_scan_id`, and `drift_type`.

In [ ]:
# SOLUTION: DriftEvent DynamoDB item design
# Access patterns:
#   1. All drift events for a scan  -> Query PK=DRIFT#SCAN#{scan_id}
#   2. Drift history for a control  -> Query PK=DRIFT#CTRL#{control_id}
#   3. All regressions (alerting)   -> GSI with drift_type as key

print('DriftEvent fields:', list(DriftEvent.__dataclass_fields__.keys()))
print()

# Example: store one DriftEvent as a DynamoDB item
example_drift = DriftEvent(
    drift_id=generate_drift_id(),
    control_id='AC-2',
    control_title='Account Management',
    previous_status='FAIL',
    current_status='PASS',
    drift_type='IMPROVEMENT',
    severity='LOW',
    previous_scan_id=scan1.scan_id,
    current_scan_id=scan2.scan_id,
    timestamp='2024-01-15T10:30:00Z',
    affected_resources=['arn:aws:iam::123456789012:root'],
    details='Control moved from FAIL to PASS after root MFA enabled',
)

drift_item = {
    # Primary key: group by scan for 'all drift in this scan' queries
    'PK': f'DRIFT#SCAN#{example_drift.current_scan_id}',
    'SK': f'CTRL#{example_drift.control_id}#{example_drift.drift_id}',
    # GSI: group by control for 'drift history of AC-2' queries
    'GSI1PK': f'DRIFT#CTRL#{example_drift.control_id}',
    'GSI1SK': example_drift.timestamp,
    **example_drift.to_dict(),
}

print('=== DriftEvent -> DynamoDB Item ===')
for k in ['PK', 'SK', 'GSI1PK', 'GSI1SK', 'drift_type', 'severity', 'previous_status', 'current_status']:
    print(f'  {k}: {drift_item[k]}')

print(f'\nQuery: "All drift in scan 2" -> Query(PK=DRIFT#SCAN#{scan2.scan_id})')
print(f'Query: "AC-2 drift history"   -> Query(GSI1PK=DRIFT#CTRL#AC-2, SK range)')

## Lab 4.2: Drift Detection with src/drift/detector.py

**Goal:** Use the real `DriftDetector` from `src/drift/detector.py` to compare our two scans.

The detector compares two `List[ControlAssessment]` and produces `List[DriftEvent]`.
This is the **Change Data Capture** pattern from DDIA Ch. 11: compare old snapshot vs new snapshot, emit events for every change.

In [ ]:
# Use the REAL DriftDetector from src/drift/detector.py
# API: detector.detect(previous_assessments, current_assessments) -> List[DriftEvent]

detector = DriftDetector()
drift_events = detector.detect(assessments1, assessments2)

print(f'Drift events detected: {len(drift_events)}')
print(f'  Regressions:  {sum(1 for d in drift_events if d.is_regression)}')
print(f'  Improvements: {sum(1 for d in drift_events if d.drift_type == "IMPROVEMENT")}')
print()

print(f'{"Control":<12} {"Type":<16} {"Before":<14} {"After":<14} {"Severity":<10}')
print('-' * 66)
for d in drift_events:
    print(f'  {d.control_id:<10} {d.drift_type:<16} {d.previous_status:<14} '
          f'{d.current_status:<14} {d.severity:<10}')

---
### What DriftDetector.detect() Does — Step by Step

Open `src/drift/detector.py` and trace through `detect()` as you read this. The method does four things:

**Step 1 — Build lookup dicts from both assessment lists:**
```python
prev_by_id = {a.control_id: a for a in previous}   # {"AC-2": ControlAssessment, ...}
curr_by_id = {a.control_id: a for a in current}     # {"AC-2": ControlAssessment, ...}
```
This is the dict-comprehension-for-fast-lookup pattern from Phase 2. Converting O(n) search into O(1) lookup. Without this, comparing 25 controls against 25 controls is 25×25 = 625 comparisons. With dicts: 25 lookups.

**Step 2 — Get the union of all control IDs from both scans:**
```python
all_ids = set(prev_by_id.keys()) | set(curr_by_id.keys())
```
`|` on sets = union. Controls that appear in *either* scan. This handles the edge case where a new control was added to the catalog between scans (it'll appear in `curr` but not `prev`) or a control was removed (it'll appear in `prev` but not `curr`).

**Step 3 — Compare each control:**
```python
for control_id in all_ids:
    prev = prev_by_id.get(control_id)   # None if control is new
    curr = curr_by_id.get(control_id)   # None if control was removed
    
    prev_status = prev.status.value if prev else "NOT_ASSESSED"
    curr_status = curr.status.value if curr else "NOT_ASSESSED"
    
    if prev_status == curr_status:
        continue   # no change — skip
    
    # Status changed → create a DriftEvent
    drift_type = classify_drift(prev_status, curr_status)
    events.append(DriftEvent(...))
```

**Step 4 — Classify the drift type:**
```python
# REGRESSION: control got worse (PASS→FAIL, PASS→PARTIAL, PARTIAL→FAIL)
# IMPROVEMENT: control got better (FAIL→PASS, FAIL→PARTIAL, PARTIAL→PASS)
# NEW_FINDING: control now has evidence it didn't before (NOT_ASSESSED→anything)
# RESOLVED: control no longer has evidence (anything→NOT_ASSESSED)
# STATUS_CHANGE: anything else
```

**The `is_regression` field is the most important one for alerting:** only regressions trigger immediate SNS alerts. Improvements are logged but don't wake anyone up at 2am.

---

In [ ]:
# Inspect a single DriftEvent in detail
# Every field comes from src/models.DriftEvent — no inline redefinition

if drift_events:
    d = drift_events[0]
    print(f'=== DriftEvent Detail ===')
    print(f'drift_id:         {d.drift_id}')
    print(f'control_id:       {d.control_id}')
    print(f'control_title:    {d.control_title}')
    print(f'previous_status:  {d.previous_status}')
    print(f'current_status:   {d.current_status}')
    print(f'drift_type:       {d.drift_type}')
    print(f'severity:         {d.severity}')
    print(f'is_regression:    {d.is_regression}')
    print(f'previous_scan_id: {d.previous_scan_id}')
    print(f'current_scan_id:  {d.current_scan_id}')
    print(f'affected_resources: {d.affected_resources}')
    print(f'details:          {d.details}')
    print()
    print('Serialized (for DynamoDB/API):')
    print(json.dumps(d.to_dict(), indent=2, default=str)[:500])
else:
    print('No drift events detected')

### Exercise 4.2: Analyze Drift Trends

Write a function `analyze_drift_trend(drift_events)` that:
1. Groups drift events by `drift_type` (REGRESSION, IMPROVEMENT, etc.)
2. Counts events per type
3. Identifies the highest-severity regression (if any)
4. Returns a summary dict

**Hint:** Use `d.drift_type`, `d.severity`, and `SEVERITY_WEIGHTS[d.severity]` for comparison.

In [ ]:
# SOLUTION: Drift trend analysis
from collections import Counter

def analyze_drift_trend(events):
    """Analyze drift events and return a summary."""
    by_type = Counter(d.drift_type for d in events)
    
    # Find highest-severity regression
    regressions = [d for d in events if d.is_regression]
    worst_regression = None
    if regressions:
        worst_regression = max(regressions,
                               key=lambda d: SEVERITY_WEIGHTS.get(d.severity, 0))
    
    return {
        'total_events': len(events),
        'by_type': dict(by_type),
        'regressions': len(regressions),
        'worst_regression': {
            'control_id': worst_regression.control_id,
            'severity': worst_regression.severity,
            'details': worst_regression.details,
        } if worst_regression else None,
    }

trend = analyze_drift_trend(drift_events)
print('Drift Trend Analysis:')
print(f'  Total events: {trend["total_events"]}')
print(f'  By type: {trend["by_type"]}')
print(f'  Regressions: {trend["regressions"]}')
if trend['worst_regression']:
    print(f'  Worst regression: {trend["worst_regression"]["control_id"]} '
          f'({trend["worst_regression"]["severity"]})')
else:
    print('  No regressions detected (good!)')

## Lab 4.3: SNS Alert Formatting

**Goal:** Format `DriftEvent` objects as SNS alert messages.
We won't call SNS (requires AWS credentials), but we'll build the exact message format that a Lambda would publish.

In [ ]:
# Format DriftEvent as an SNS alert message
# In production: a Lambda triggered by DynamoDB Streams would call sns.publish()

def format_drift_alert(drift_event: DriftEvent) -> dict:
    """Format a DriftEvent as an SNS publish() payload."""
    subject = (f'[{drift_event.severity}] Compliance Drift: '
              f'{drift_event.control_id} ({drift_event.drift_type})')
    
    message = f"""
COMPLIANCE DRIFT ALERT
======================

Control:   {drift_event.control_id} — {drift_event.control_title}
Type:      {drift_event.drift_type}
Severity:  {drift_event.severity}
Timestamp: {drift_event.timestamp}

Status Change: {drift_event.previous_status} -> {drift_event.current_status}
{drift_event.details}

Affected Resources ({len(drift_event.affected_resources)}):
{chr(10).join(f'  - {r}' for r in drift_event.affected_resources) or '  (none)'}

Action Required:
  1. Review the configuration change in AWS Console
  2. Remediate if this is an unintended regression
  3. Document the change for audit trail

Scan: {drift_event.previous_scan_id} -> {drift_event.current_scan_id}
"""
    
    return {
        'TopicArn': 'arn:aws:sns:us-east-1:123456789012:compliance-alerts',
        'Subject': subject[:100],  # SNS subject max 100 chars
        'Message': message.strip(),
    }

# Demo: format alerts for all regressions
regressions = [d for d in drift_events if d.is_regression]
print(f'Formatting {len(regressions)} regression alert(s)...\n')

for d in regressions:
    alert = format_drift_alert(d)
    print(f'Subject: {alert["Subject"]}')
    print(f'TopicArn: {alert["TopicArn"]}')
    print(f'Message preview:')
    print(alert['Message'][:400])
    print()

if not regressions:
    # Show an improvement alert instead
    improvements = [d for d in drift_events if d.drift_type == 'IMPROVEMENT']
    if improvements:
        alert = format_drift_alert(improvements[0])
        print(f'Subject: {alert["Subject"]}')
        print(alert['Message'][:400])

---
### SNS Alert Design — Every Decision Has a Reason

Before reading the `format_drift_alert()` function, understand *why* it's structured the way it is.

**SNS constraints to design around (DVA-C02 exam):**
```
Subject:          max 100 characters — used as email subject line
Message:          max 256 KB — effectively unlimited for text
MessageAttributes: key-value metadata for filtering at subscription level
TopicArn:         must be in the same region as the SNS client
```

**The `Subject[:100]` truncation:**
```python
'Subject': subject[:100]
# Why? SNS rejects subjects over 100 chars with an InvalidParameterException
# The :100 slice is a hard requirement, not a style choice
```

**The fan-out pattern — one topic, many subscribers:**
```
SNS Topic: compliance-drift-alerts
    │
    ├── Email subscription → compliance@company.com
    ├── Email subscription → ciso@company.com
    ├── Lambda subscription → create-jira-ticket Lambda
    ├── SQS subscription → audit-log queue
    └── HTTPS subscription → Slack webhook
```
One `sns.publish()` call delivers to all subscribers simultaneously. This is the **fan-out pattern** from System Design interviews. The scanner never knows or cares how many downstream systems consume the alert. Adding a new integration (e.g., PagerDuty) is zero code in the scanner — just add a subscription in the AWS Console.

**Message format for human vs machine consumption:**
```python
# Human-readable: formatted text for email (what we built)
# Machine-readable: JSON for Lambda processing
# Best practice: include BOTH using MessageStructure='json'

message = json.dumps({
    "default":  format_text_alert(drift_event),   # email/SMS
    "lambda":   json.dumps(drift_event.to_dict()), # Lambda
    "sqs":      json.dumps(drift_event.to_dict()), # SQS consumer
})
sns.publish(..., Message=message, MessageStructure='json')
```
We use plain text here for readability, but production systems use `MessageStructure='json'` so Lambda subscribers get clean JSON while email subscribers get formatted text.

**`chr(10).join(...)` — what is `chr(10)`?**  
`chr(10)` is the newline character `\n`. You can't put a literal `\n` inside an f-string expression, so `chr(10)` is the workaround when you need a newline inside a comprehension. It's a Python quirk, not a readability choice.

---

### Terraform Configuration for SNS Topic

```hcl
resource "aws_sns_topic" "compliance_alerts" {
  name = "compliance-drift-alerts"
}

resource "aws_sns_topic_subscription" "compliance_email" {
  topic_arn = aws_sns_topic.compliance_alerts.arn
  protocol  = "email"
  endpoint  = "compliance@example.com"
}

resource "aws_sns_topic_subscription" "compliance_slack" {
  topic_arn = aws_sns_topic.compliance_alerts.arn
  protocol  = "https"
  endpoint  = "https://hooks.slack.com/services/YOUR/WEBHOOK/URL"
}
```


## Lab 4.4: DynamoDB Streams & Lambda Triggers

**Goal:** Understand how DynamoDB Streams trigger automatic drift detection when new scan results are written.

**Architecture:**
```
Scan Lambda writes ControlAssessment items to DynamoDB
  ↓
DynamoDB Stream event (NEW_AND_OLD_IMAGES)
  ↓
Lambda trigger reads old + new item
  ↓
If status changed: create DriftEvent, publish to SNS
```

The Lambda handler below uses the same `src/` classes. In production, it reads `ControlAssessment.from_dict()` to deserialize the DynamoDB stream record.

In [ ]:
# Lambda handler for DynamoDB Streams drift detection
# This is what would run in AWS Lambda — shown here as a string for study

lambda_handler_code = '''
import json, os, boto3
from src.models import ControlAssessment, ControlStatus, DriftEvent, generate_drift_id
from src.drift.detector import DriftDetector

sns = boto3.client("sns")
SNS_TOPIC = os.environ["SNS_TOPIC_ARN"]

def handler(event, context):
    """Triggered by DynamoDB Stream when ControlAssessment items are written."""
    results = []
    
    for record in event["Records"]:
        if record["eventName"] != "MODIFY":
            continue  # Only check updates (status changes)
        
        old_image = record["dynamodb"]["OldImage"]
        new_image = record["dynamodb"]["NewImage"]
        
        old_status = old_image.get("status", {}).get("S", "")
        new_status = new_image.get("status", {}).get("S", "")
        
        if old_status == new_status:
            continue  # No drift
        
        control_id = new_image["control_id"]["S"]
        
        # Classify drift type
        if old_status == "PASS" and new_status in ("FAIL", "PARTIAL"):
            drift_type = "REGRESSION"
            severity = "CRITICAL"
        elif old_status in ("FAIL", "PARTIAL") and new_status == "PASS":
            drift_type = "IMPROVEMENT"
            severity = "LOW"
        else:
            drift_type = "STATUS_CHANGE"
            severity = "MEDIUM"
        
        # Publish SNS alert for regressions
        if drift_type == "REGRESSION":
            sns.publish(
                TopicArn=SNS_TOPIC,
                Subject=f"[{severity}] Drift: {control_id} {old_status}->{new_status}",
                Message=f"Control {control_id} regressed from {old_status} to {new_status}"
            )
        
        results.append({"control_id": control_id, "drift_type": drift_type})
    
    return {"statusCode": 200, "body": json.dumps(results)}
'''

print('Lambda handler for DynamoDB Streams drift detection:')
print(lambda_handler_code)
print('Key insight: The handler uses src/models classes (ControlAssessment, DriftEvent)')
print('and the same drift classification logic as src/drift/detector.py')

---
### DynamoDB Streams — The Three-Phase Lambda Pattern

The Lambda handler above processes a **DynamoDB Stream event**. Every time an item in the compliance table is created, updated, or deleted, DynamoDB appends a record to its stream. Lambda reads those records in batches. This is the same mechanism as Kafka consumers reading from a topic (DDIA Ch. 11, p. 448).

**What a stream record looks like (raw):**
```json
{
  "eventName": "MODIFY",
  "dynamodb": {
    "OldImage": {
      "PK":     {"S": "CTRL#AC-2"},
      "SK":     {"S": "SCAN#2024-01-08"},
      "status": {"S": "FAIL"}
    },
    "NewImage": {
      "PK":     {"S": "CTRL#AC-2"},
      "SK":     {"S": "SCAN#2024-01-15"},
      "status": {"S": "PASS"}
    }
  }
}
```
The `{"S": "FAIL"}` format is DynamoDB's type descriptor syntax. `"S"` = String, `"N"` = Number, `"BOOL"` = Boolean, `"L"` = List, `"M"` = Map. Every value is wrapped in a `{"type": value}` envelope. The boto3 DynamoDB `resource` client handles this automatically with `TypeDeserializer` — the `client` (lower-level) does not.

**The three phases every stream Lambda must handle:**

```python
# Phase 1: Filter — ignore events you don't care about
if record["eventName"] != "MODIFY":
    continue   # INSERT and REMOVE don't trigger drift checks

# Phase 2: Extract — pull the fields you need from OldImage and NewImage
old_status = record["dynamodb"]["OldImage"].get("status", {}).get("S", "")
new_status = record["dynamodb"]["NewImage"]["status"]["S"]

# Phase 3: Decide — if relevant, act; otherwise skip
if old_status == new_status:
    continue   # no status change → no drift → no alert
```

**`NEW_AND_OLD_IMAGES` — why both? Why not just `NEW_IMAGE`?**  
To detect drift you need to compare. If you only stored the new image, you'd know the current state but not what it changed *from*. `OLD_IMAGE` tells you the before state. `NEW_AND_OLD_IMAGES` stores both. The extra storage cost is minimal. Use `KEYS_ONLY` when you just need to know something changed (e.g., cache invalidation); use `NEW_AND_OLD_IMAGES` when you need to know how it changed (drift detection, audit logs, undo).

**Idempotency edge case:**  
Lambda + DynamoDB Streams is "at-least-once delivery." Your handler may receive the same stream record twice (Lambda retry on failure). The handler must be idempotent: calling it twice must produce the same result as calling it once. The defensive approach: before publishing the SNS alert, check if a DriftEvent with this `drift_id` already exists in DynamoDB. If yes, skip. This is the same pattern as the Lambda handler idempotency check from Phase 3.

---

### Terraform Configuration for Lambda Trigger

```hcl
resource "aws_lambda_event_source_mapping" "compliance_stream" {
  event_source_arn  = aws_dynamodb_table.compliance_data.stream_arn
  function_name     = aws_lambda_function.drift_detector.function_name
  enabled           = true
  batch_size        = 10
  start_position    = "LATEST"
}

resource "aws_lambda_function" "drift_detector" {
  function_name = "drift-detector"
  role          = aws_iam_role.lambda_role.arn
  handler       = "index.handler"
  runtime       = "python3.11"
  filename      = "drift_detector.zip"

  environment {
    variables = {
      SNS_TOPIC_ARN = aws_sns_topic.compliance_alerts.arn
    }
  }
}
```


## Lab 4.5: EventBridge for Real-Time AWS Config Changes

**Goal:** Monitor AWS Config for configuration changes and trigger drift detection in real-time.

**Architecture:**

```
Someone modifies S3 bucket
  ↓
AWS Config detects change
  ↓
Config Change event
  ↓
EventBridge Rule matches event
  ↓
Trigger Lambda: UpdateComplianceStatus
  ↓
Lambda calls AWS Config API
  ↓
Updates DynamoDB with new status
  ↓
DynamoDB Stream triggers DriftDetector
  ↓
Alert published to SNS
```


### EventBridge Rule Example

```hcl
# Match AWS Config configuration changes
resource "aws_cloudwatch_event_rule" "config_changes" {
  name        = "config-compliance-changes"
  description = "Trigger on AWS Config compliance changes"

  event_pattern = jsonencode({
    source      = ["aws.config"]
    detail-type = ["Config Rules – Compliance Change"]
    detail = {
      newEvaluationResult = {
        complianceType = ["NON_COMPLIANT"]
      }
    }
  })
}

# Target: Lambda function that updates DynamoDB
resource "aws_cloudwatch_event_target" "update_compliance" {
  rule      = aws_cloudwatch_event_rule.config_changes.name
  target_id = "UpdateComplianceDB"
  arn       = aws_lambda_function.update_compliance.arn
}

# Lambda permission to be invoked by EventBridge
resource "aws_lambda_permission" "allow_eventbridge" {
  statement_id  = "AllowExecutionFromEventBridge"
  action        = "lambda:InvokeFunction"
  function_name = aws_lambda_function.update_compliance.function_name
  principal     = "events.amazonaws.com"
  source_arn    = aws_cloudwatch_event_rule.config_changes.arn
}
```

### Lambda Function Triggered by EventBridge

```python
import boto3
import json

config_client = boto3.client('config')
dynamodb = boto3.resource('dynamodb')
table = dynamodb.Table('compliance-assessments')

def handler(event, context):
    # EventBridge event from AWS Config
    config_rule_name = event['detail']['configRuleName']
    compliance_type = event['detail']['newEvaluationResult']['complianceType']
    resource_id = event['detail']['resourceId']
    timestamp = event['time']
    
    # Map Config rule to control ID
    control_id = config_rule_to_control(config_rule_name)
    
    # Update DynamoDB with new status
    table.put_item(Item={
        'PK': f'CONTROL#{control_id}',
        'SK': f'SCAN#{timestamp}',
        'status': compliance_type,
        'timestamp': timestamp,
        'source': 'AWS_CONFIG'
    })
    
    return {'statusCode': 200, 'message': 'Updated'}
```


---
### EventBridge vs DynamoDB Streams — When to Use Which

You now have two different ways to trigger drift detection. This is a design choice you will be asked about in system design interviews and on DVA-C02. Here is the exact decision table:

| Dimension | DynamoDB Streams | EventBridge (AWS Config) |
|---|---|---|
| **Trigger source** | Write to your DynamoDB table | AWS service emits an event (S3 change, Config rule, IAM policy update) |
| **Latency** | Milliseconds after the write | Seconds to minutes (Config evaluation lag) |
| **What you detect** | Changes to data your app controls | Changes to the actual AWS infrastructure |
| **Ordering guarantee** | Per-partition ordered | Best-effort (no ordering guarantee) |
| **Replay** | Up to 24 hours lookback | No built-in replay |
| **Cost model** | Per-shard (always running) | Per event emitted |
| **Use when** | Your app writes the state | AWS infrastructure writes the state |

**The critical conceptual difference:**

```
DynamoDB Streams:
  Your scanner → writes assessment → DynamoDB → Stream → Lambda (drift check)
  You control both ends of the pipe.

EventBridge (AWS Config):
  An engineer changes a Security Group → AWS Config → EventBridge → Lambda (drift check)
  AWS infrastructure changes trigger your system. You don't control the source.
```

**Why the compliance system uses BOTH:**

DynamoDB Streams catches drift *in your compliance data* — for example, if two scan Lambdas run in parallel and write conflicting assessment results. EventBridge catches drift *in the actual AWS resources* — for example, an engineer manually disabling MFA on the root account at 11pm on a Friday.

```
Full coverage:
                  ┌─────────────────────────────────┐
                  │  EventBridge (infrastructure)    │
                  │  Detects: IAM changes,           │
                  │  S3 policy changes, SG changes   │
                  │  → Lambda → DynamoDB write       │
                  └────────────────┬────────────────┘
                                   │
                                   ▼
                            DynamoDB Table
                                   │
                  ┌────────────────▼────────────────┐
                  │  DynamoDB Streams (data layer)   │
                  │  Detects: assessment status      │
                  │  changes (PASS → FAIL)           │
                  │  → Lambda → SNS alert            │
                  └─────────────────────────────────┘
```

**DVA-C02 exam pattern to know:**  
EventBridge + Lambda requires an explicit `aws_lambda_permission` granting `events.amazonaws.com` the right to invoke the function. This is easy to forget. DynamoDB Streams + Lambda uses an event source mapping (not a resource policy) — the Lambda execution role needs `dynamodb:GetRecords`, `dynamodb:GetShardIterator`, `dynamodb:DescribeStream`, and `dynamodb:ListStreams` permissions instead.

**DDIA connection (Ch. 11 — Stream Processing, p. 444):**  
Kleppmann distinguishes between *pull-based* consumers (Kafka/Streams: consumer polls for new records) and *push-based* consumers (webhooks/EventBridge: source pushes to consumer). DynamoDB Streams is pull-based — Lambda polls the shard. EventBridge is push-based — AWS Config pushes the event. Pull gives you flow control and ordering. Push gives you lower latency. For compliance, we use pull for the data layer (ordering matters for drift history) and push for infrastructure events (latency matters for real-time alerting).

---

## Final Exercises

### Exercise 4.3: Build a Comprehensive Drift Report

Create a function `generate_drift_report(drift_events, posture1, posture2)` that:
1. Groups drift events by severity (CRITICAL, HIGH, MEDIUM, LOW)
2. Calculates the compliance percentage delta between scans
3. Lists the top 3 most urgent regressions
4. Returns a structured dict ready for email/PDF formatting

Use the real `DriftEvent` and `CompliancePosture` objects from earlier in this notebook.

### Exercise 4.4: Implement Drift Suppression

Some drift is expected (auto-scaling, temporary debugging). Design a suppression system:
1. A `suppress_drift(control_id, reason, duration_hours)` function that creates a DynamoDB item with TTL
2. A `is_suppressed(control_id)` check before alerting
3. Store suppressions with key pattern `PK=SUPPRESS#CTRL#{control_id}`, `SK=timestamp`

How does DynamoDB TTL handle the automatic cleanup? (Hint: items are deleted during compaction, similar to LSM tree cleanup from DDIA Ch. 3.)

### Exercise 4.5: DAX Caching Layer

Add **DAX (DynamoDB Accelerator)** for hot data:
```hcl
resource "aws_dax_cluster" "compliance_cache" {
  cluster_name       = "compliance-cache"
  iam_role_arn       = aws_iam_role.dax.arn
  node_type          = "dax.r4.large"
  replication_factor = 3
}
```

Questions:
1. Which access patterns benefit most from DAX? (Hint: `GetItem` for posture is called on every dashboard load.)
2. What's the consistency trade-off? (DAX serves eventually consistent reads by default.)
3. How does this relate to DDIA Ch. 5's discussion of replication lag?

### Exercise 4.6: Batch Write Assessments to DynamoDB

Write a function that takes `List[ControlAssessment]` and writes them to DynamoDB using `batch_write_item()`.
DynamoDB limits batch writes to 25 items. How do you handle our 24-control catalog? What if it grows to 50 controls?

```python
def batch_write_assessments(table, assessments: list):
    # YOUR CODE: chunk into groups of 25, call batch_write_item()
    pass
```

---

---
## Before You Write Exercise Code — Patterns to Memorize

The exercises below ask you to use `batch_write_item`, DynamoDB pagination, and drift suppression. These are boilerplate patterns that every AWS engineer reaches for from muscle memory. Read this section before starting the exercises.

---

### Pattern 1: DynamoDB batch_write — The 25-Item Chunk Problem

`batch_write_item` is limited to 25 items per call. This is a hard AWS limit. Any function that writes a list of unknown length must chunk it:

```python
def batch_write(table, items: list):
    # Chunk into groups of 25 (DynamoDB hard limit per batch_write call)
    chunk_size = 25
    for i in range(0, len(items), chunk_size):
        chunk = items[i : i + chunk_size]
        with table.batch_writer() as batch:
            for item in chunk:
                batch.put_item(Item=item)
```

**Why `range(0, len(items), chunk_size)`?**
```
items = [a, b, c, d, e, f, g]  # 7 items
chunk_size = 3

range(0, 7, 3) = [0, 3, 6]     # start indices

items[0:3] = [a, b, c]          # chunk 1
items[3:6] = [d, e, f]          # chunk 2
items[6:9] = [g]                # chunk 3 (Python doesn't error on out-of-bounds slices)
```

**`table.batch_writer()` as context manager** — boto3's `batch_writer()` buffers writes and automatically retries unprocessed items. You don't need to manually handle `UnprocessedItems`. Always prefer `batch_writer()` over calling `batch_write_item()` directly for this reason.

---

### Pattern 2: DynamoDB Pagination — `LastEvaluatedKey`

DynamoDB `query()` and `scan()` return at most 1 MB of data per call. If the result set is larger, the response includes `LastEvaluatedKey`. You must loop until this key is absent:

```python
def query_all(table, pk_value: str) -> list:
    """Fetch all items for a PK, handling DynamoDB pagination."""
    items = []
    kwargs = {
        "KeyConditionExpression": Key("PK").eq(pk_value)
    }
    
    while True:
        response = table.query(**kwargs)
        items.extend(response["Items"])
        
        # DynamoDB signals "more data" by including LastEvaluatedKey
        last_key = response.get("LastEvaluatedKey")
        if not last_key:
            break  # no more pages — we're done
        
        # Pass the key back as ExclusiveStartKey to get the next page
        kwargs["ExclusiveStartKey"] = last_key
    
    return items
```

**The `while True` / `break` pattern for pagination:**  
This is idiomatic Python for "loop until a condition you discover mid-loop." The alternative (checking the condition in the `while`) requires fetching the first page before the loop, then re-checking. The `while True` / `break` form keeps the logic in one place.

**When this matters:** A compliance table with 50 controls × 52 weekly scans = 2,600 items per year. At ~200 bytes each, that's ~520 KB — right at the 1 MB boundary. Pagination is not a theoretical concern here; you will hit it within 18 months of production use.

---

### Pattern 3: DynamoDB `ConditionExpression` for Idempotent Writes

When writing a drift event or suppression record, you want to write it only if it doesn't already exist (idempotency). Use `attribute_not_exists`:

```python
import boto3
from botocore.exceptions import ClientError

def write_if_not_exists(table, item: dict):
    """Write item only if PK+SK doesn't already exist (idempotent put)."""
    try:
        table.put_item(
            Item=item,
            ConditionExpression="attribute_not_exists(PK)"
            # ConditionExpression is checked atomically — no race condition
        )
        return True   # written
    except ClientError as e:
        if e.response["Error"]["Code"] == "ConditionalCheckFailedException":
            return False  # already existed — that's fine, idempotent
        raise  # unexpected error — re-raise
```

**Why `attribute_not_exists(PK)` and not a read-then-write?**  
Reading an item and then writing it only if it was absent has a race condition: two Lambdas could both read "not found" and both write. `ConditionExpression` is evaluated atomically by DynamoDB — only one writer wins, the other gets `ConditionalCheckFailedException`. This is the same guarantee as a SQL `INSERT ... WHERE NOT EXISTS`.

---

### Pattern 4: DynamoDB TTL for Automatic Suppression Expiry

TTL lets DynamoDB delete items automatically after a timestamp. Used in Exercise 4.4 for drift suppressions:

```python
from datetime import datetime, timedelta, timezone

def ttl_from_hours(hours: int) -> int:
    """Return Unix timestamp for 'now + hours', suitable for DynamoDB TTL."""
    expiry = datetime.now(timezone.utc) + timedelta(hours=hours)
    return int(expiry.timestamp())

# Usage:
suppression_item = {
    "PK": f"SUPPRESS#CTRL#{control_id}",
    "SK": datetime.now(timezone.utc).isoformat(),
    "reason": reason,
    "ttl": ttl_from_hours(24),  # auto-delete after 24 hours
}
```

**Why `int()` and not `float()`?**  
DynamoDB TTL expects a Number attribute containing a Unix timestamp as an integer (seconds). `datetime.timestamp()` returns a float (e.g., `1714849800.123`). `int()` truncates to seconds. DynamoDB silently ignores TTL attributes that are not integers, so the `int()` cast is load-bearing, not cosmetic.

**The 48-hour lag caveat:**  
TTL deletion is handled during DynamoDB's background compaction. Items are *eligible* for deletion at the TTL timestamp but may not be *actually deleted* for up to 48 hours. For drift suppression, this means a suppression with a 1-hour TTL might still be suppressing alerts 49 hours later. Add an explicit `if item.get("ttl", 0) > time.time()` check in `is_suppressed()` to enforce the boundary in application code.

---

### Relevant Reading for the Exercises

| Topic | Resource | Why It Matters |
|---|---|---|
| `batch_write_item` limits | [DynamoDB BatchWriteItem](https://docs.aws.amazon.com/amazondynamodb/latest/APIReference/API_BatchWriteItem.html) | 25-item limit, `UnprocessedItems` handling |
| DynamoDB pagination | [DynamoDB Query pagination](https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/Query.Pagination.html) | `LastEvaluatedKey` loop pattern |
| TTL | [DynamoDB TTL](https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/TTL.How_To_Enable_TTL.html) | 48-hour delete lag, integer requirement |
| ConditionExpression | [Conditional writes](https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/Expressions.OperatorsAndFunctions.html) | `attribute_not_exists`, idempotent writes |
| boto3 batch_writer | [boto3 batch_writer](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/dynamodb.html#batch-writing) | Automatic retry of `UnprocessedItems` |
| DAX caching | [DAX developer guide](https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/DAX.html) | Cache hit/miss, eventual consistency trade-off |

---

## Summary

In this phase, you learned:

1. **Configuration Drift** — Why it matters for compliance, detection strategies (batch vs event-driven vs hybrid)
2. **DynamoDB Single-Table Design** — PK/SK patterns for ControlAssessment, CompliancePosture, DriftEvent
3. **Drift Detection** — Using `src/drift/detector.py` to compare scan results and produce DriftEvent objects
4. **SNS Alerting** — Formatting DriftEvent as SNS messages for auditor notifications
5. **DynamoDB Streams** — Lambda triggers for real-time drift detection on writes
6. **EventBridge** — Real-time AWS Config change monitoring
7. **S3 Lifecycle Policies** — STANDARD → GLACIER → DEEP_ARCHIVE → Delete for evidence retention

### Key classes used (all from `src/`)
| Class | Module | Purpose |
|-------|--------|---------|
| `ControlAssessment` | `src.models` | Stored in DynamoDB, compared for drift |
| `CompliancePosture` | `src.models` | Aggregate stats stored in DynamoDB |
| `DriftEvent` | `src.models` | Drift changelog, triggers SNS alerts |
| `DriftDetector` | `src.drift.detector` | `detect(prev, curr) -> List[DriftEvent]` |
| `ControlMappingEngine` | `src.mapper.engine` | Produces assessments from scan results |

### DDIA connections
- **Ch. 3 (Storage Engines):** LSM trees → DynamoDB internals, compaction, TTL cleanup
- **Ch. 5 (Replication):** Multi-AZ durability, eventual vs strong consistency, DAX cache lag
- **Ch. 11 (Stream Processing):** DynamoDB Streams as CDC, EventBridge for real-time events

### DVA-C02 connections
- **DynamoDB:** Single-table design, GSIs, TTL, Streams, batch operations, DAX (Domain 2)
- **EventBridge:** Event patterns, rule targets, Lambda permissions (Domain 2)
- **SNS:** Topic subscriptions, message formatting, fan-out pattern (Domain 2)
- **S3:** Lifecycle policies, storage classes, Glacier retrieval tiers (Domain 2)

**Next: Phase 5** — Executive Dashboard, API Gateway, and serving this data to users.

---

---
## What a Senior Engineer Would Take From This Phase

After four phases of building this compliance system, you now have the full picture. Before moving to Phase 5, here is what a senior engineer internalizes — not the syntax, but the *reasoning*.

---

### 1. Every Design Decision Is a Trade-Off Between Access Patterns and Flexibility

The single-table DynamoDB design you built in Lab 4.1 is optimized for exactly three queries. If a fourth query is needed later, you will either add a GSI (cost: money and write amplification) or do a full table scan (cost: latency and $$$). SQL gives you flexibility at the cost of JOIN performance at scale. DynamoDB gives you speed at the cost of upfront design discipline.

**The senior engineer's instinct:** Before touching the schema, ask "what are all the ways we will ever query this data?" Write them down. Then design the PK/SK and GSIs to serve those exact patterns. Revisiting a DynamoDB table schema in production is painful — there is no `ALTER TABLE`.

---

### 2. Drift Detection Is a CDC Problem, Not a Diff Problem

You might have framed DriftDetector as "compare two lists." But the senior engineer frames it as a CDC (Change Data Capture) problem: "what state changes do I need to surface as events?" This reframing matters because it connects to a much larger design space — Kafka, Debezium, DynamoDB Streams, EventBridge — all of which are CDC systems. Once you recognize the pattern, you can evaluate the right tool for the access frequency, ordering requirements, and replay needs.

**The senior engineer's instinct:** When you see "compare state A vs state B," ask "is this a batch CDC job or a streaming CDC job?" Batch is simpler and cheaper. Streaming adds latency guarantees but complexity. Choose streaming only when detection delay has a real cost (e.g., a CRITICAL regression needs a page, not a daily email).

---

### 3. Lambda + DynamoDB Streams + SNS Is a Well-Known Pattern — Know Its Failure Modes

This architecture is used in hundreds of production systems. You should know its failure modes before you deploy it:

```
Failure Mode 1: Lambda retry storm
  Cause: Lambda fails to process a stream record → DynamoDB retries → Lambda fails again
  Result: the stream gets stuck at that record; new records are blocked
  Fix: Dead Letter Queue (DLQ) on the event source mapping + BisectBatchOnFunctionError

Failure Mode 2: SNS delivery failure
  Cause: Email subscription bounces, HTTPS endpoint is down
  Result: alert lost silently
  Fix: SNS Dead Letter Queue per subscription, CloudWatch alarm on NumberOfNotificationsFailed

Failure Mode 3: Idempotency gap
  Cause: Lambda succeeds after writing DriftEvent but before publishing SNS
  Result: the event is in DynamoDB but the alert was never sent
  Fix: publish SNS first, then write DriftEvent (if the write fails, SNS already sent — acceptable)
       OR use DynamoDB Transactions to make both writes atomic
```

**The senior engineer's instinct:** Design for failure first. Every async integration (Lambda triggers, SNS, EventBridge) has a failure mode. Know the failure mode before you write the happy path.

---

### 4. TTL Is "Approximately" Deletes — Never Rely on It for Security Guarantees

DynamoDB TTL will delete your items within ~48 hours of expiry. "Within 48 hours" is a service best-effort guarantee. For evidence retention under FedRAMP (3 years minimum), TTL ensures *old items are eventually cleaned up* — it doesn't ensure items are accessible *until exactly* day 1095. For drift suppression, items may suppress alerts for up to 48 hours after the suppression was supposed to expire. For security-sensitive expiry (e.g., session tokens), use application-level expiry checks in addition to TTL, not instead of.

---

### 5. The Four Labs Form One End-to-End Pipeline You Can Apply to Any Domain

```
Lab 4.1 (DynamoDB design)    ← Any domain that needs time-series state storage
Lab 4.2 (Drift detection)    ← Any domain that compares snapshots (infra, config, prices, inventory)
Lab 4.3 (SNS alerting)       ← Any domain that needs fan-out notifications
Lab 4.4 + 4.5 (Streams/EventBridge) ← Any domain that needs real-time reaction to state changes
```

You built this for NIST 800-53 compliance checks. The same architecture applies to:
- **Infrastructure cost monitoring:** detect when a new resource type pushes spend above a threshold
- **Inventory drift:** detect when a product price changes unexpectedly
- **Security posture monitoring:** detect when a new CVE affects a running container image
- **Configuration management:** detect when a Kubernetes ConfigMap differs from the GitOps repo

The classes change. The pattern doesn't.

---

### Boilerplate Checklist — Phase 4 Patterns to Have in Your Back Pocket

```python
# DynamoDB chunked batch write
for i in range(0, len(items), 25):
    with table.batch_writer() as batch:
        for item in items[i:i+25]:
            batch.put_item(Item=item)

# DynamoDB paginated query
items, kwargs = [], {"KeyConditionExpression": Key("PK").eq(pk)}
while True:
    r = table.query(**kwargs)
    items.extend(r["Items"])
    if not (last := r.get("LastEvaluatedKey")): break
    kwargs["ExclusiveStartKey"] = last

# Idempotent write
try:
    table.put_item(Item=item, ConditionExpression="attribute_not_exists(PK)")
except ClientError as e:
    if e.response["Error"]["Code"] != "ConditionalCheckFailedException": raise

# TTL timestamp
ttl = int((datetime.now(timezone.utc) + timedelta(hours=n)).timestamp())

# DynamoDB type descriptor → plain value
value = record["dynamodb"]["NewImage"]["status"]["S"]   # {"S": "FAIL"} → "FAIL"

# SNS subject hard limit
subject = full_subject[:100]

# chr(10) for newline in f-string expression
lines = chr(10).join(f"  - {r}" for r in resource_list)
```

---

## Additional Resources

- [DynamoDB Developer Guide](https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/)
- [DynamoDB Streams](https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/Streams.html)
- [AWS Config Rules](https://docs.aws.amazon.com/config/latest/developerguide/managed-rules-by-aws-config.html)
- [EventBridge Rules](https://docs.aws.amazon.com/eventbridge/latest/userguide/eb-rules.html)
- [SNS Publisher Guide](https://docs.aws.amazon.com/sns/latest/dg/PublishTopic.html)
- [DynamoDB Single-Table Design](https://aws.amazon.com/blogs/database/single-table-design-with-amazon-dynamodb/)
